# MMJL First Smoke Test

This notebook is intended to be run with JupyterLab launched from the
repository root.

For this test run, the expected launch pattern is:

```powershell
cd C:\David\my_repos_dwb\multimodal-jupy-logger
.\.venv_test_mmjl\Scripts\Activate.ps1
jupyter lab
```

The first setup code cell assumes that:

```python
Path.cwd()
```

is the repository root, so that:

```python
Path.cwd() / "src"
```

points to the package source directory.

If this notebook is opened from `examples/` or another working directory,
update the setup cell so that `repo_root` points at the actual repository
root.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
##endof:  if str(src_path) not in sys.path

print("repo_root:", repo_root)
print("src_path:", src_path)
print("python:", sys.executable)

repo_root: C:\David\my_repos_dwb\multimodal-jupy-logger
src_path: C:\David\my_repos_dwb\multimodal-jupy-logger\src
python: C:\David\my_repos_dwb\multimodal-jupy-logger\.venv_test_mmjl\Scripts\python.exe


In [2]:
from multimodal_jupy_logger import (
    MultimodalJupyLogger,
    register_jupy_logger,
    jupy_logger_register,
)

logger = MultimodalJupyLogger(root="mmlj_test_log")
logger.inspect_manifest()

Manifest: C:\David\my_repos_dwb\multimodal-jupy-logger\mmlj_test_log\manifest.tsv
Root: C:\David\my_repos_dwb\multimodal-jupy-logger\mmlj_test_log
Artifacts: C:\David\my_repos_dwb\multimodal-jupy-logger\mmlj_test_log\artifacts
Rows: 0

By kind:

By MIME:


[]

In [3]:
register_jupy_logger()

[DONE] Registered: %jupy_save, %jupy_file, %%jupy_log, %jupy_markdown, %jupy_html, %jupy_inspect, %jupy_validate


## Smoke-test note

This notebook currently tests the lean-to development workflow:

1. Add `src/` to `sys.path`.
2. Import the package.
3. Instantiate a logger.
4. Register Jupyter magics.
5. Use `%%jupy_log`, `%jupy_inspect`, `%jupy_validate`,
   `%jupy_markdown`, and `%jupy_html`.

The notebook is an example artifact, not the canonical install path.
Later versions should support an editable install or package entry point.

In [4]:
%%jupy_log --label details-summary-tip --mime text/markdown
<details>
<summary>Click to see something hidden</summary>

Here is something hidden.

</details>

[DONE] Logged text: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\1781480251_2026-06-14T193731-0400_details-summary-tip.md


WindowsPath('C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/artifacts/1781480251_2026-06-14T193731-0400_details-summary-tip.md')

In [5]:
%jupy_inspect
print("\n-----\n")
%jupy_validate

Manifest: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
Root: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log
Artifacts: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts
Rows: 1

By kind:
  text: 1

By MIME:
  text/markdown: 1

-----

Manifest: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\manifest.tsv
Rows checked: 1
Missing artifacts: 0


[]

In [6]:
%jupy_markdown
%jupy_html

[DONE] Markdown timeline written: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\timeline.md
[DONE] HTML timeline written: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\timeline.html


WindowsPath('C:/David/my_repos_dwb/multimodal-jupy-logger/jupy_log/timeline.html')

In [7]:
# Lean-to tree

import pathlib

#  Other option
#+
#+ -----------------------------------
#+   !pip install seedir
#+ -----------------------------------
#+   import seedir as sd
#+   sd.seedir()
#+ -----------------------------------
#+
#+ But staying lean-to for now.

print(f"Current working directory:\n{pathlib.Path.cwd()}")



def path_part_has_match(
      path_part: str,
      exclusion_items: list[str],
    ) -> bool:
    '''
    Return True if any exclusion string is a substring of one path part.
    '''

    return any(
        item in path_part
        for item in exclusion_items
    )
##endof:  path_part_has_match(path_part, exclusion_items)



def any_path_part_has_match(
      path_parts: tuple[str, ...],
      exclusion_items: list[str],
    ) -> bool:
    '''
    Return True if any exclusion string matches any path component.
    '''

    return any(
        path_part_has_match(
            path_part,
            exclusion_items,
        )
        for path_part in path_parts
    )
##endof:  any_path_part_has_match(path_parts, exclusion_items)



def dwb_py_tree(
      this_dir: pathlib.Path=pathlib.Path.cwd(),
      indent_length: int=4,
      dirs_to_exclude: list[str] | None=None,
      files_to_exclude: list[str] | None=None,
    ) -> None:
    '''
    Quick-and-reckless version of tree, see directory contents
    hierarchically.

    For exclusions, a simple in (for substring) is used for the test.

    Directory exclusions are checked against relative directory path
    parts only, not against the full absolute path. This prevents names
    like C:/Users/.../jamesloganhowlett from accidentally excluding
    wanted project files.

    File exclusions are checked against the filename only.
    '''

    child_item_path = None
    current_depth = 0
    dir_parts_to_check = ()
    exists_match_for_dir = False
    exists_match_for_file = False
    relative_path = None
    suffix = ""
    this_child = ""
    this_indent = ""
    this_is_dir = False
    this_is_excluded_dir = False
    this_is_excluded_file = False
    this_is_file = False

    if dirs_to_exclude is None:
        dirs_to_exclude = []
    ##endof:  if dirs_to_exclude is None

    if files_to_exclude is None:
        files_to_exclude = []
    ##endof:  if files_to_exclude is None

    this_dir = pathlib.Path(this_dir).expanduser().resolve()

    print(f"+ {this_dir}")

    for child_item_path in sorted(this_dir.rglob("*")):
        relative_path = child_item_path.relative_to(this_dir)
        this_child = child_item_path.name

        this_is_dir = child_item_path.is_dir()
        this_is_file = child_item_path.is_file()

        if this_is_dir:
            dir_parts_to_check = relative_path.parts
        else:
            dir_parts_to_check = relative_path.parts[:-1]
        ##endof:  if this_is_dir

        exists_match_for_dir = any_path_part_has_match(
            dir_parts_to_check,
            dirs_to_exclude,
        )
        this_is_excluded_dir = exists_match_for_dir

        if this_is_excluded_dir:
            continue
        ##endof:  if this_is_excluded_dir

        exists_match_for_file = path_part_has_match(
            this_child,
            files_to_exclude,
        )
        this_is_excluded_file = this_is_file and exists_match_for_file

        if this_is_excluded_file:
            continue
        ##endof:  if this_is_excluded_file

        suffix = "/" if this_is_dir else ""
        current_depth = len(relative_path.parts)
        this_indent = " " * indent_length * current_depth

        print(f"{this_indent}+ {this_child}{suffix}")
    ##endof:  for child_item_path in sorted(this_dir.rglob("*"))
##endof:  dwb_py_tree(...)

print("\n\nHere is the tree:\n")

dwb_py_tree(dirs_to_exclude=[".git", "pycache", "venv", "ipynb_checkpoints"])

Current working directory:
C:\David\my_repos_dwb\multimodal-jupy-logger


Here is the tree:

+ C:\David\my_repos_dwb\multimodal-jupy-logger
    + .gitattributes
    + .gitignore
    + docs/
        + context_documents/
        + dev_notes/
    + examples/
        + complete_pre_jupyter_lab_powershell_io_2026-06-14T152800-0400.log
        + test_setup_and_start.md
    + jupy_log/
        + artifacts/
            + 1781480251_2026-06-14T193731-0400_details-summary-tip.md
        + manifest.tsv
        + timeline.html
        + timeline.md
    + LICENSE
    + mmlj_test_log/
        + artifacts/
        + manifest.tsv
    + README.md
    + reproducibility_win_init__requirements.txt
    + requirements-ml.txt
    + requirements.txt
    + src/
        + multimodal_jupy_logger/
            + __init__.py
            + logger.py
            + magics.py
            + metadata.py
            + mmjl_cli.py
    + Untitled.ipynb


In [8]:
def dwb_line_counter(
      filename: pathlib.Path | str,
      do_print: bool=False
    ) -> int:
    
    num_lines = None
    with open(filename, "rb") as file_handle:
        num_lines = sum(1 for _ in file_handle)
    ##endof:  with open ... file_handle

    if do_print:  print(f"{filename} has {num_lines} lines")

    return num_lines
##endof:  dwb_line_counter(filename, do_print)

def dwb_file_printer(filename: pathlib.Path | str) -> None:
    with open(filename, 'r') as file_handle:
        print(file_handle.read())
    ##endof:  with open ... file_handle
##endof:  dwb_file_printer(filename)

### Numbers of lines in relevant files.

In [9]:
_ = dwb_line_counter(
    (
        "jupy_log/artifacts/1781480251_2026-06-14T193731-0400"
        "_details-summary-tip.md"
    ), 
    do_print=True)

_ = dwb_line_counter("jupy_log/manifest.tsv", do_print=True)

_ = dwb_line_counter("jupy_log/timeline.html", do_print=True)

_ = dwb_line_counter("jupy_log/timeline.md", do_print=True)

_ = dwb_line_counter("mmlj_test_log/manifest.tsv", do_print=True)

jupy_log/artifacts/1781480251_2026-06-14T193731-0400_details-summary-tip.md has 6 lines
jupy_log/manifest.tsv has 2 lines
jupy_log/timeline.html has 29 lines
jupy_log/timeline.md has 30 lines
mmlj_test_log/manifest.tsv has 1 lines


`jupy_log/artifacts/1781480251_2026-06-14T193731-0400_details-summary-tip.md`

In [10]:
dwb_file_printer(
    (
        "jupy_log/artifacts/1781480251_2026-06-14T193731-0400"
        "_details-summary-tip.md"
    )
)

<details>
<summary>Click to see something hidden</summary>

Here is something hidden.

</details>



`jupy_log/manifest.tsv`

In [11]:
dwb_file_printer("jupy_log/manifest.tsv")

timestamp	kind	mime	label	path
1781480251_2026-06-14T193731-0400	text	text/markdown	details-summary-tip	C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\artifacts\1781480251_2026-06-14T193731-0400_details-summary-tip.md



`jupy_log/timeline.html`

In [12]:
dwb_file_printer("jupy_log/timeline.html")

<html>
<body>
<h1>Jupyter Log Timeline</h1>
<pre>
**Multimodal Jupy Logger HTML Timeline**

This log file: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\timeline.html
Timestamp: 1781480262_2026-06-14T193742-0400
Machine, user, etc.: DESKTOP-O7KM5A5 Anast@DESKTOP-O7KM5A5

---

---

---

</pre>
<hr>
<h3>details-summary-tip</h3>
<p><code>1781480251_2026-06-14T193731-0400</code> â€” text â€” text/markdown</p>
<pre>&lt;details&gt;
&lt;summary&gt;Click to see something hidden&lt;/summary&gt;

Here is something hidden.

&lt;/details&gt;
</pre>
</body>
</html>


`jupy_log/timeline.md`

In [13]:
dwb_file_printer("jupy_log/timeline.md")

# Jupyter Log Timeline

**Multimodal Jupy Logger Markdown Timeline**

This log file: C:\David\my_repos_dwb\multimodal-jupy-logger\jupy_log\timeline.md
Timestamp: 1781480262_2026-06-14T193742-0400
Machine, user, etc.: DESKTOP-O7KM5A5 Anast@DESKTOP-O7KM5A5

---

---

---


---

## details-summary-tip

`1781480251_2026-06-14T193731-0400` â€” `text` â€” `text/markdown`

```
<details>
<summary>Click to see something hidden</summary>

Here is something hidden.

</details>

```



`mmlj_test_log/manifest.tsv`

In [14]:
dwb_file_printer("mmlj_test_log/manifest.tsv")

timestamp	kind	mime	label	path

